<a href="https://colab.research.google.com/github/fuadfach/geog761lab/blob/main/geog761_lab1_fuad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Set up GEE API
import ee
ee.Authenticate()
ee.Initialize(project='geog761-ffac001')

In [2]:
# Install the geemap package (only needs to be run once, uncomment below and run it the first time you run this notebook in a session).
#!pip install geemap

In [3]:
import geemap
import os

# **Loading a map and displaying satellite data**

(1) Find and add the basemap 'OpenTopoMap' to the display window AND change the code so that the map opens and zooms to the city of Auckland.
Provide a link to your code in a notebook and the output as a figure in the answer proforma. (2 pts)

In [36]:
# Specify a different kind of basemap to display data layers over
Map = geemap.Map(center=(-36.8485, 174.7633), zoom=14) #<- note the lat-lon coordinate pair here
Map.add_basemap("OpenTopoMap")
Map

Map(center=[-36.8485, 174.7633], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

(3) Correct the display of the Landsat 7 image so that it displays in 'True Colour'. Provide a notebook example of your code and the corrected output over the whole of New Zealand. (10 pts)

In [37]:
# Add Earth Engine datasets to our map by first creating variables to hold the calls to the EE api
Map = geemap.Map(center=(-41.0182, 175.3623), zoom=5)
dem = ee.Image("USGS/SRTMGL1_003")
landcover = ee.Image("ESA/GLOBCOVER_L4_200901_200912_V2_3").select("landcover")
landsat7 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")
states = ee.FeatureCollection("TIGER/2018/States")

In [38]:
# Set visualization parameters.
vis_params = {
    "min": 0,
    "max": 4000, #<- if your satellite image is all white or all black, these vis params are the first thing to check and change
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"], #<- these are HTML colour codes
}

In [39]:
# Add a variety of different Earth Engine layers to the Map object
Map.addLayer(dem, vis_params, "SRTM DEM", True, 0.5) #<- note vis params called from the dictionary we set up before
Map.addLayer(landcover, {}, "Land cover")
Map.addLayer(
    landsat7, {"bands": ["B3", "B2", "B1"], "min": 0, "max": 100}, "Landsat 7" #<- note vis params in a dictionary here inside the add layer call
)
Map.addLayer(states, {}, "US States")

In [40]:
# Dump all of this into the map view and take a look...
# Scroll around (look at both NZ and USA), check and uncheck the layers in the layer menu on the top-right.
Map

Map(center=[-41.0182, 175.3623], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

# Cloud cover
(7) Produce a table that tells me the average amount of cloud cover per LandSat image in a year: for 2005, 2010 and 2015. Do this over a region of your choice, as a percentage.
Provide a notebook example of your code that does so, in addition to your table. (5 pts)


In [35]:
import pandas as pd

# Define small region around Alor Island, Indonesia
point = ee.Geometry.Point([124.518393, -8.217483])
region = point.buffer(500).bounds()

years = [2005, 2010, 2015]
summary_data = []

for year in years:
    # Filter collection for the region and calendar year
    col = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
           .filterBounds(region)
           .filterDate(f"{year}-01-01", f"{year}-12-31"))

    # Scene-wide average metadata cloud cover
    avg_scene_cloud = col.aggregate_mean('CLOUD_COVER').getInfo()
    total_images = col.size().getInfo()

    # Calculate ROI-specific cloud percentage per scene
    def compute_roi_cloud_pct(img):
        qa = img.select('QA_PIXEL')
        # Bit 3 = Cloud, Bit 4 = Cloud Shadow
        cloud_or_shadow = qa.bitwiseAnd(1 << 3).neq(0).Or(qa.bitwiseAnd(1 << 4).neq(0))

        # Mean reducer yields fraction of cloudy pixels in ROI
        roi_stats = cloud_or_shadow.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=30,
            maxPixels=1e9
        )
        return img.set('roi_cloud_pct', ee.Number(roi_stats.get('QA_PIXEL')).multiply(100))

    # Calculate average ROI cloud coverage across all scenes in the year
    col_with_stats = col.map(compute_roi_cloud_pct)
    avg_roi_cloud = col_with_stats.aggregate_mean('roi_cloud_pct').getInfo()

    summary_data.append({
        'Year': year,
        'Total Scenes': total_images,
        'Avg Scene Cloud Cover (%)': round(avg_scene_cloud, 2) if avg_scene_cloud else 0,
        'Avg ROI Cloud Cover (%)': round(avg_roi_cloud, 2) if avg_roi_cloud else 0
    })

# Output as DataFrame
results_df = pd.DataFrame(summary_data)
print(results_df)

   Year  Total Scenes  Avg Scene Cloud Cover (%)  Avg ROI Cloud Cover (%)
0  2005            41                      24.27                    24.41
1  2010            38                      33.29                    37.48
2  2015            43                      22.60                    29.78


In [29]:
# Function to mask clouds using QA_PIXEL band
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    # Bits 3 and 4 are cloud and cloud shadow respectively
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0).And(
                 qa.bitwiseAnd(1 << 4).eq(0))
    return image.updateMask(cloud_mask)

# Apply mask
masked_image = mask_clouds(image)

In [30]:
# Report Image Quality Over ROI
# Total pixels in region
pixel_area = ee.Image.pixelArea().clip(region)

# Count valid pixels before and after masking
total_pixels = pixel_area.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=region,
    scale=30,
    maxPixels=1e9
).getInfo()['area']

valid_pixels = masked_image.select('SR_B3').reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=region,
    scale=30,
    maxPixels=1e9
).getInfo()['SR_B3']

cloud_coverage_pct = 100 * (1 - valid_pixels / total_pixels)

print(f"Total Pixels: {total_pixels}")
print(f"Valid (non-cloud) Pixels: {valid_pixels}")
print(f"Estimated Cloud Coverage in ROI: {cloud_coverage_pct:.2f}%")

Total Pixels: 1122
Valid (non-cloud) Pixels: 1103
Estimated Cloud Coverage in ROI: 1.69%


In [32]:
# Visualize the before and after
Map = geemap.Map(center=[-8.217483, 124.518393], zoom=12)

vis = {
    'bands': ['SR_B3', 'SR_B2', 'SR_B1'],
    'min': 0,
    'max': 30000,
    'gamma': 1.4
}

Map.addLayer(image, vis, 'Original Image (Cloudy)')
Map.addLayer(masked_image, vis, 'Cloud Masked')
Map

Map(center=[-8.217483, 124.518393], controls=(WidgetControl(options=['position', 'transparent_bg'], position='…